# SATD Classifier Training
**Model:** Linear SVM (Selected per architecture rationale)
**Dataset:** SATDAUG (`data-augmentation-code_comments.csv`)

## Step 1: Dataset Loading
This step loads the augmented dataset and verifies the class distribution.

In [1]:
import pandas as pd

# Load SATDAUG Dataset
df = pd.read_csv('../../data/raw/data-augmentation-code_comments.csv', skiprows=1)
df = df.dropna(subset=['text', 'classification'])
print(f"Loaded dataset with {len(df):,} comments.")
print("\nClass Distribution:")
print(df['classification'].value_counts())


Loaded dataset with 68,512 comments.

Class Distribution:
classification
non_debt              58204
code/design_debt       2703
documentation_debt     2701
test_debt              2635
requirement_debt       2269
Name: count, dtype: int64


## Step 2: Preprocessing & Vectorization
Here we split the dataset into an 80/20 train/test split and extract our TF-IDF features using the parameters specified in the documentation (ngrams 1-2, 25k max features).

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Prepare Features and Labels
X = df['text'].astype(str)
y = df['classification'].astype(str)

# Stratified split to preserve the class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Set: {len(X_train):,} samples")
print(f"Test Set:     {len(X_test):,} samples")

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=25000,
    stop_words='english'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Feature matrix shape: {X_train_tfidf.shape}")


Training Set: 54,809 samples
Test Set:     13,703 samples


Feature matrix shape: (54809, 25000)


## Step 3 & 4: Model Training, Evaluation, and Export
Here we train the LinearSVC model on our TF-IDF features, evaluate its performance per class, and finally export the full pipeline (Vectorizer + Classifier) to our `models/` directory.

In [3]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
import joblib
import os
import time

# Create and train the full pipeline
pipeline = Pipeline([
    ('tfidf', vectorizer),
    ('clf', LinearSVC(class_weight='balanced', random_state=42, max_iter=2000))
])

print("Training model...")
t0 = time.time()
pipeline.fit(X_train, y_train)
print(f"Training complete in {time.time() - t0:.2f}s")

# Evaluate per class (Task: Evaluate the classifier per class, with counts)
y_pred = pipeline.predict(X_test)
print("\nClassification Report (per class, with counts):")
print(classification_report(y_test, y_pred))

# Export the model
os.makedirs('../../models', exist_ok=True)
model_path = '../../models/satd_v1.joblib'
joblib.dump(pipeline, model_path)
print(f"\nSuccessfully exported production SATD pipeline to {model_path}")


Training model...


Training complete in 4.20s

Classification Report (per class, with counts):


                    precision    recall  f1-score   support

  code/design_debt       0.71      0.73      0.72       541
documentation_debt       0.99      0.99      0.99       540
          non_debt       0.99      0.98      0.99     11641
  requirement_debt       0.83      0.85      0.84       454
         test_debt       0.97      1.00      0.98       527

          accuracy                           0.97     13703
         macro avg       0.90      0.91      0.90     13703
      weighted avg       0.97      0.97      0.97     13703




Successfully exported production SATD pipeline to ../../models/satd_v1.joblib
